In [0]:
from pyspark.sql.functions import *

In [0]:
%sql
create database if not exists mydb;

In [0]:
%sql
describe database extended mydb;

database_description_item,database_description_value
Catalog Name,workspace
Namespace Name,mydb
Comment,
Location,
Owner,ayushhathiwan2@gmail.com
Properties,
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)


In [0]:
%sql
-- creating a watermark table which will be store the watermark value for each table
create table if not exists mydb.watermark_metadata(
    table_name string,
    watermark_column string, 
    watermark_value timestamp 
)

In [0]:
source = "employees"
source_path = "/Volumes/workspace/mydb/myvolume/employee_src/"
target = "employees_report"
target_path = "/Volumes/workspace/mydb/myvolume/employee_tgt/"

In [0]:

# watermark date for the table
watermark_df = (spark.table("mydb.watermark_metadata").filter(col("table_name") == source)
                .select("watermark_value"))
watermark_date = watermark_df.collect()[0][0]
watermark_date

datetime.datetime(2026, 7, 14, 2, 1, 45, 29000)

In [0]:
# reading source where last updated timestamp is greater than watermark date
emp_df = (spark.read.csv(path=f"{source_path}/{source_file}", header=True, inferSchema=True)
          .filter(col("LAST_UPDATED") > watermark_date))

emp_df.display()

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID,LAST_UPDATED
208,Priya,Verma,PVERMA,515.123.9992,11-JAN-25,FI_ACCOUNT,9200,0.0,108,100,2026-07-14T02:41:48.536Z
207,Aman,Sharma,ASHARMA,515.123.9991,10-JAN-25,IT_PROG,8500,0.0,103,60,2026-07-14T02:41:48.536Z


In [0]:
# applying transformation and writing data
emp_df = emp_df.fillna({"COMMISSION_PCT": 0}).dropna(subset=["MANAGER_ID"])

emp_df.write.mode("append").save(f"{target_path}/{target}")


EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID,LAST_UPDATED
208,Priya,Verma,PVERMA,515.123.9992,11-JAN-25,FI_ACCOUNT,9200,0.0,108,100,2026-07-14T02:41:48.536Z
207,Aman,Sharma,ASHARMA,515.123.9991,10-JAN-25,IT_PROG,8500,0.0,103,60,2026-07-14T02:41:48.536Z


In [0]:
df = spark.range(11)

df.display()

id
0
1
2
3
4
5
6
7
8
9


In [0]:
# find the max value of last updated column and update the watermark
new_watermark_value = emp_df.agg(max("LAST_UPDATED")).collect()[0][0]

spark.sql(f"""
UPDATE mydb.watermark_metadata
SET watermark_value = '{new_watermark_value}'
WHERE table_name = '{source}'
""")

datetime.datetime(2026, 7, 14, 2, 41, 48, 536000)

In [0]:
# update th


DataFrame[num_affected_rows: bigint]

In [0]:
%sql
select * from mydb.watermark_metadata;

table_name,watermark_column,watermark_value
employees,last_updated,2026-07-14T02:41:48.536Z
